In [1]:
import pandas as pd
import pathlib

base = pathlib.Path("input")
use_cases = ["bee-use-case", "cloud-use-case", "marine-use-case", "termite-use-case"]
display_names = {
    "bee-use-case": "Bee",
    "cloud-use-case": "Cloud",
    "marine-use-case": "Marine",
    "termite-use-case": "Termite",
}

def extract_rank(classification, rank):
    """Extract a GTDB rank from a classification string."""
    prefix_map = {"phylum": "p__", "genus": "g__", "species": "s__"}
    prefix = prefix_map.get(rank)
    if pd.isna(classification) or not prefix:
        return None
    for part in str(classification).split(";"):
        part = part.strip()
        if part.startswith(prefix):
            name = part[len(prefix):].strip()
            return name if name else "unclassified"
    return "unclassified"

rows = []
top30_tables = {}

for uc in use_cases:
    dat = base / uc / "data"

    drep = pd.read_csv(dat / "drep.csv")
    drep["genome"] = drep["genome"].str.replace(r"\.fasta$", "", regex=True)

    total_mags = len(drep)
    n_clusters = drep["secondary_cluster"].nunique()
    cluster_sizes = drep.groupby("secondary_cluster").size()
    multi_member = (cluster_sizes >= 2).sum()

    checkm = pd.read_csv(dat / "checkm2.tsv", sep="\t")
    checkm["Name"] = checkm["Name"].str.replace(r"\.fasta$", "", regex=True)

    gtdb = pd.read_csv(dat / "gtdb.tsv", sep="\t")
    gtdb["user_genome"] = gtdb["user_genome"].str.replace(r"\.fasta$", "", regex=True)

    merged = pd.merge(checkm, gtdb, left_on="Name", right_on="user_genome", how="left")

    merged["species"] = merged["classification"].apply(
        lambda x: extract_rank(x, "species")
    )
    merged.loc[merged["species"].isna(), "species"] = "no GTDB hit"

    species_counts = merged["species"].value_counts()
    n_unclassified = species_counts.get("unclassified", 0)
    n_no_gtdb = species_counts.get("no GTDB hit", 0)
    unclassified_pct = (n_unclassified + n_no_gtdb) / len(merged) * 100

    top20 = species_counts.head(20)
    top20_str = "; ".join(f"{s} ({n})" for s, n in top20.items())

    comp_avg = merged["Completeness"].mean()
    comp_min = merged["Completeness"].min()
    comp_max = merged["Completeness"].max()
    cont_avg = merged["Contamination"].mean()
    cont_min = merged["Contamination"].min()
    cont_max = merged["Contamination"].max()

    rows.append({
        "Use case": display_names[uc],
        "Total MAGs": total_mags,
        "Species-level clusters": n_clusters,
        "Clusters with \u22652 members": multi_member,
        "Unclassified species": f"{n_unclassified + n_no_gtdb} / {len(merged)} ({unclassified_pct:.0f}%)",
        "Completeness": f"{comp_avg:.1f}% ({comp_min:.1f}–{comp_max:.1f})",
        "Contamination": f"{cont_avg:.2f}% ({cont_min:.2f}–{cont_max:.2f})",
        "Top 20 species (cluster reps)": top20_str,
    })

    # --- Top 30 cluster reps with phylum, genus, species ---
    merged["phylum"] = merged["classification"].apply(lambda x: extract_rank(x, "phylum"))
    merged["genus"] = merged["classification"].apply(lambda x: extract_rank(x, "genus"))

    top30 = merged.sort_values("Completeness", ascending=False).head(30).copy()
    top30 = top30[["Name", "phylum", "genus", "species", "Completeness", "Contamination"]]
    top30.columns = ["MAG", "Phylum", "Genus", "Species", "Completeness", "Contamination"]
    top30["Completeness"] = top30["Completeness"].round(1).astype(str) + "%"
    top30["Contamination"] = top30["Contamination"].round(2).astype(str) + "%"
    top30_tables[display_names[uc]] = top30.reset_index(drop=True)

summary = pd.DataFrame(rows)
summary = summary[["Use case", "Total MAGs", "Species-level clusters",
                  "Clusters with \u22652 members", "Unclassified species",
                  "Completeness", "Contamination", "Top 20 species (cluster reps)"]]
summary

,Use case,Total MAGs,Species-level clusters,Clusters with ≥2 members,Unclassified species,Completeness,Contamination,Top 20 species (cluster reps)
0,Bee,161,38,22,6 / 38 (16%),90.6% (75.0–100.0),2.47% (0.05–18.57),unclassified (6); Frischella perrara (1); Gill...
1,Cloud,230,215,7,213 / 215 (99%),16.1% (1.7–100.0),2.92% (0.00–26.55),unclassified (213); Mycobacterium sp002013775 ...
2,Marine,153,132,18,116 / 132 (88%),91.2% (76.6–100.0),2.71% (0.00–21.21),unclassified (116); Olleya sediminilitoris (1)...
3,Termite,33,26,2,9 / 14 (64%),88.3% (72.0–100.0),2.86% (0.13–9.97),unclassified (9); JAIRMT01 sp031258505 (1); JA...


In [2]:
# Top 30 cluster reps per use case (sorted by completeness)
for name, tbl in top30_tables.items():
    print(f"\n=== {name} ===")
    print(tbl.to_string(index=False))


=== Bee ===
                   MAG         Phylum               Genus                        Species Completeness Contamination
 SRR24759608_bin_64934 Pseudomonadota         Providencia      Providencia alcalifaciens       100.0%         0.05%
 SRR24759606_bin_47167 Pseudomonadota         Providencia         Providencia rettgeri_D       100.0%         0.37%
 SRR24759616_bin_33137 Pseudomonadota             Proteus              Proteus mirabilis       100.0%          0.1%
 SRR24759605_bin_35353 Pseudomonadota        Arsenophonus           Arsenophonus apicola       100.0%          0.7%
 SRR24759598_bin_98484 Pseudomonadota          Klebsiella             Klebsiella oxytoca       100.0%         0.21%
SRR24759598_bin_132977 Pseudomonadota            Serratia            Serratia ureilytica       100.0%         1.81%
 SRR24759606_bin_22995 Pseudomonadota         Entomomonas                   unclassified       100.0%         0.53%
 SRR24759603_bin_51590 Pseudomonadota         Entomomonas  